# Урок 23. Обработка табличных данных в Python

11 класс · III четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [← Урок 22](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-22.ipynb) · [Урок 24 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-24.ipynb)

---

Загрузка CSV, структура таблицы, отбор строк и столбцов, новые вычисляемые поля. Пропуски и грязные данные.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 11А
Класс = "" #@param {type:"string"}

import importlib, urllib.request
for адрес in ("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py",
              "https://cdn.jsdelivr.net/gh/LeksssGray/informatika-7-11@main/lib/schoolinf.py"):
    try:
        urllib.request.urlretrieve(адрес, "schoolinf.py")
        break
    except Exception as ошибка:
        print("⚠️ Не скачалась библиотека проверки:", ошибка)
else:
    raise SystemExit("Проверь интернет и запусти эту ячейку ещё раз. "
                     "Не помогло — покажи это сообщение учителю.")
import schoolinf as si
importlib.reload(si)
si.start(lesson="11-23", name=ФИО, klass=Класс)

## Разбираемся

### Откуда берутся данные

Почти все данные в мире лежат в таблицах, а переносят их между
программами в формате **CSV** — «значения, разделённые запятыми».
Это обычный текстовый файл:

```
фамилия,класс,информатика,физика
Иванов,9А,92,78
Петрова,9Б,64,88
```

Первая строка — заголовки, остальные — записи. Разделителем
бывает запятая, точка с запятой или табуляция: в русской локали
Excel по умолчанию ставит **точку с запятой**, и это первое,
на чём спотыкаются.

### Почему не просто split

Соблазнительно разобрать CSV так:

```python
поля = строка.split(",")
```

И это сломается на первой же строке вида
`"Иванов, Иван",9А,92`: запятая внутри кавычек — часть значения,
а не разделитель.

Поэтому есть модуль `csv` из стандартной библиотеки. Он знает
про кавычки, переводы строк внутри полей и разделители.

```python
import csv

with open("данные.csv", encoding="utf-8", newline="") as файл:
    for запись in csv.DictReader(файл):
        print(запись["фамилия"], запись["информатика"])
```

`DictReader` отдаёт каждую строку словарём, где ключи — заголовки
столбцов. Обращаться по имени столбца гораздо надёжнее, чем
по номеру: добавили столбец в середину — код не сломался.

### Всё приходит строками

Главная особенность CSV: **типов в нём нет**. Число 92 приедет
строкой `"92"`, и складывать такие значения бессмысленно.
Преобразование — ваша работа:

```python
балл = int(запись["информатика"])
```

### Грязные данные

В настоящей выгрузке встречается всё:

| Проблема | Как выглядит | Что делать |
|---|---|---|
| пропуск | пустая строка `""` | пропустить запись или подставить значение |
| пробелы | `" 92 "` | `.strip()` |
| запятая вместо точки | `"4,5"` | заменить на точку |
| текст в числовом поле | `"н/д"`, `"-"` | пропустить с подсчётом |
| дубликаты | две одинаковые записи | убрать по ключу |

Правило: **считать количество пропущенного**. Молча выбрасывать
плохие записи нельзя — если их треть, выводы будут неверными,
и об этом нужно знать.

### Схема обработки

Почти любая задача с данными укладывается в четыре шага:

1. **загрузить** — прочитать файл;
2. **очистить** — привести типы, выбросить мусор, посчитать потери;
3. **посчитать** — отбор, группировка, агрегаты;
4. **показать** — таблица, график, вывод словами.

Сегодня первые два шага плюс простые расчёты. Группировка —
на следующем уроке.

### Вычисляемые поля

Часто нужен столбец, которого в файле нет: сумма баллов,
средний балл, «сдал или нет». Его считают при загрузке
и кладут в ту же запись:

```python
запись["сумма"] = балл1 + балл2
```

Дальше работают с ним, как с обычным полем.

## Смотрим, как это работает

### Пример 1. Делаем файл с данными

Настоящего файла у нас нет — создадим его прямо здесь, заодно
нарочно испортив часть записей.

In [ ]:
import csv

содержимое = """фамилия,класс,информатика,физика
Иванов,9А,92,78
Петрова,9Б,64,88
Сидоров,9А, 71 ,65
Кузнецова,9Б,95,
Орлов,9А,н/д,72
Никитина,9Б,83,91
Иванов,9А,92,78
"""

with open("данные.csv", "w", encoding="utf-8", newline="") as файл:
    файл.write(содержимое)

print("Файл записан, строк:", len(содержимое.strip().split("\n")))

### Пример 2. Чтение через DictReader

In [ ]:
with open("данные.csv", encoding="utf-8", newline="") as файл:
    записи = list(csv.DictReader(файл))

print("Записей:", len(записи))
print("Столбцы:", list(записи[0].keys()))
print()
for запись in записи[:3]:
    print(запись)

Каждая запись — словарь. Заметьте: все значения строки, даже там,
где явно числа.

### Пример 3. Очистка с подсчётом потерь

In [ ]:
чистые = []
потеряно = 0
причины = {}


def годится(значение):
    """Можно ли из этого получить число."""
    текст = (значение or "").strip().replace(",", ".")
    return текст != "" and текст.replace(".", "", 1).isdigit()


def в_число(значение):
    return float((значение or "").strip().replace(",", "."))


for запись in записи:
    если_оба = годится(запись["информатика"]) and годится(запись["физика"])
    if not если_оба:
        потеряно += 1
        причина = "пустое поле" if not (запись["информатика"] and запись["физика"]) else "не число"
        причины[причина] = причины.get(причина, 0) + 1
        continue
    инф = в_число(запись["информатика"])
    физ = в_число(запись["физика"])
    чистые.append({
        "фамилия": запись["фамилия"],
        "класс": запись["класс"],
        "информатика": инф,
        "физика": физ,
        "сумма": инф + физ,
    })

print(f"Загружено: {len(записи)}, пригодных: {len(чистые)}, потеряно: {потеряно}")
print("Причины:", причины)

### Пример 4. Дубликаты

In [ ]:
видели = set()
без_дублей = []
дублей = 0

for запись in чистые:
    ключ = (запись["фамилия"], запись["класс"])
    if ключ in видели:
        дублей += 1
        continue
    видели.add(ключ)
    без_дублей.append(запись)

print(f"Было {len(чистые)}, дубликатов {дублей}, осталось {len(без_дублей)}")

Ключ здесь — пара «фамилия, класс». Выбор ключа — содержательное
решение: если в школе два Иванова из 9А, такой ключ склеит их
в одного.

### Пример 5. Отбор и расчёты

In [ ]:
сильные = [з for з in без_дублей if з["информатика"] >= 80]

print("Набрали 80 и выше по информатике:")
for з in сильные:
    print(f"  {з['фамилия']:<12} {з['информатика']:>5.0f} + {з['физика']:>5.0f} = {з['сумма']:>5.0f}")

баллы = [з["информатика"] for з in без_дублей]
print()
print(f"Средний по информатике: {sum(баллы) / len(баллы):.2f}")
print(f"Максимум: {max(баллы):.0f}, минимум: {min(баллы):.0f}")

### Пример 6. Запись результата обратно в CSV

In [ ]:
with open("отчёт.csv", "w", encoding="utf-8", newline="") as файл:
    писатель = csv.DictWriter(файл, fieldnames=["фамилия", "класс", "сумма"])
    писатель.writeheader()
    for з in без_дублей:
        писатель.writerow({"фамилия": з["фамилия"], "класс": з["класс"], "сумма": з["сумма"]})

print(open("отчёт.csv", encoding="utf-8").read())

### Пример 7. Почему split ломается

In [ ]:
строка = '"Иванов, Иван",9А,92'

print("Через split:", строка.split(","))
print("Через csv:  ", next(csv.reader([строка])))

Первый способ разорвал фамилию пополам, второй понял кавычки.

## Пробуем сами

В задачах записи приходят списком словарей — так же, как их
отдаёт `DictReader`.

### Задача 1. Разделитель в русском Excel

Какой разделитель ставит русская версия Excel при сохранении CSV?

In [ ]:
#@title 🧩 Задача 1. Разделитель { display-mode: "form" }
#@markdown Выбери разделитель
разделитель = "выбери ответ" #@param ["выбери ответ", "запятая", "точка с запятой", "двоеточие"]

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("1", разделитель, "692a6175587b7b64",
         hint="Потому что запятая занята десятичной дробью.")

### Задача 2. Преобразование значения

Функция получает строку из CSV и возвращает число `float` или
`None`, если значение непригодно.

Убирайте пробелы по краям и считайте запятую десятичной точкой.

In [ ]:
def в_число(значение):
    return ...

In [ ]:
si.check("2", в_число, [
    (" 92 ", 92.0),
    ("4,5", 4.5),
    ("", None),
    ("н/д", None),
    ("0", 0.0),
])

### Задача 3. Сколько записей потеряно

Функция получает список записей и имя числового поля. Возвращает
количество записей, где это поле непригодно для расчётов.

Пользуйтесь функцией из предыдущей задачи.

In [ ]:
def потеряно(записи, поле):
    return ...

In [ ]:
si.check("3", потеряно, [
    (([{"б": "92"}, {"б": ""}, {"б": "н/д"}], "б"), 2),
    (([{"б": "1"}, {"б": "2"}], "б"), 0),
    (([], "б"), 0),
])

### Задача 4. Вычисляемое поле

Функция добавляет каждой записи поле `"сумма"` — сумму двух
числовых полей — и возвращает список записей.

Записи с непригодными значениями пропускайте.

In [ ]:
def добавить_сумму(записи, поле1, поле2):
    return ...

In [ ]:
si.check("4", добавить_сумму, [
    (([{"a": "1", "b": "2"}], "a", "b"), [{"a": "1", "b": "2", "сумма": 3.0}]),
    (([{"a": "1", "b": ""}], "a", "b"), []),
    (([], "a", "b"), []),
])

### Задача 5. Убрать дубликаты

Функция убирает записи, у которых совпадает значение заданного
поля, оставляя первую встреченную. Порядок сохраняется.

In [ ]:
def без_дублей(записи, поле):
    return ...

In [ ]:
si.check("5", без_дублей, [
    (([{"ф": "Иванов"}, {"ф": "Петров"}, {"ф": "Иванов"}], "ф"),
     [{"ф": "Иванов"}, {"ф": "Петров"}]),
    (([{"ф": "А"}], "ф"), [{"ф": "А"}]),
    (([], "ф"), []),
])

### Задача 6. Среднее по столбцу

Функция возвращает среднее значение числового поля по всем
пригодным записям, округлённое до двух знаков.

Если пригодных записей нет, верните 0.

In [ ]:
def среднее_по_полю(записи, поле):
    return ...

In [ ]:
si.check("6", среднее_по_полю, [
    (([{"б": "90"}, {"б": "80"}], "б"), 85.0),
    (([{"б": "90"}, {"б": "н/д"}], "б"), 90.0),
    (([{"б": "1"}, {"б": "2"}, {"б": "2"}], "б"), 1.67),
    (([{"б": ""}], "б"), 0),
])

## Домашнее задание

### Домашнее задание 1. Настоящий набор данных

Найдите открытый набор данных в CSV — погода, население, цены,
статистика спорта — хотя бы на 500 строк.

Загрузите его, очистите и напишите отчёт из пяти строк:
сколько записей загружено, сколько потеряно и почему, каковы
минимум, максимум и среднее по главному числовому полю.

Отдельно отметьте, какие именно проблемы с данными встретились.
Покажите работу учителю.

### Домашнее задание 2. Отбор по условию

Функция возвращает список записей, у которых числовое поле
не меньше порога. Непригодные записи пропускаются.

In [ ]:
def отобрать(записи, поле, порог):
    return ...

In [ ]:
si.check("дз2", отобрать, [
    (([{"б": "90"}, {"б": "50"}], "б", 80), [{"б": "90"}]),
    (([{"б": "н/д"}, {"б": "90"}], "б", 50), [{"б": "90"}]),
    (([], "б", 0), []),
])

### Домашнее задание 3. Какой модуль читает CSV

Как называется модуль стандартной библиотеки для работы
с CSV-файлами?

In [ ]:
#@title 🏠 Домашнее 3. Модуль { display-mode: "form" }
#@markdown Впиши название
модуль = "" #@param {type:"string"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("дз3", модуль, "82cacb7d3298441d",
         hint="Так же, как и сам формат.")

---

### Любопытно

Формат CSV старше почти всего, чем вы пользуетесь: он появился
в 1972 году в компиляторах IBM Fortran. Стандарта у него не было
три десятилетия — первое формальное описание вышло только
в 2005-м, и то как «информационный» документ.

Именно поэтому в CSV столько разночтений: разделители, кавычки,
переводы строк, кодировки. Формат, которому полвека и который
никто не проектировал, остаётся главным способом обмена данными
между программами — просто потому, что его понимают все.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 22](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-22.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 24 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-24.ipynb)